In [2]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.utils import bootstrap_project_paths

project_root = bootstrap_project_paths()

from src.data import load_sales
from src.pipelines import train_validate_models
from src.models import *
import pandas as pd
import numpy as np

DATA_ROOT= project_root / "data/datathon-2026-round-1"

Project root: D:\MyML\datathon2026
Source path added: D:\MyML\datathon2026\src


In [39]:
TRAIN_RANGE = ("2013-01-01","2021-12-31")  # Date range for training data
VALIDATION_RANGE = ("2022-01-01","2022-12-31")  # Date range for validation data

df = load_sales(data_root=DATA_ROOT)
# Basic time features
df["day"] = df["date"].dt.day
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year

# date of month không không thay dổi performance nhưng không thể thay thế 2 feature trên
df["day_of_week"] = df["date"].dt.dayofweek
df["week_of_year"] = df["date"].dt.isocalendar().week

# Cái này quan trọng
df["month_sin"] = np.sin(2 * np.pi * df["month"]/12)
df["month_cos"] = np.cos(2 * np.pi * df["month"]/12)

model_config = SklearnRegressorConfig(
    model_type="lightgbm"
)
results = train_validate_models(
    df,
    train_range=TRAIN_RANGE,
    predict_range=VALIDATION_RANGE,
    model_config=model_config
)
results["metrics"]

{'Revenue': {'mae': 629938.4088949503,
  'rmse': 873829.9468792224,
  'mape': 20.48176295140322,
  'smape': 21.927980964787196,
  'r2': 0.7274528916899218},
 'COGS': {'mae': 531731.2435477221,
  'rmse': 720330.5512214329,
  'mape': 20.646167479842596,
  'smape': 21.29867807790941,
  'r2': 0.7561006323767676}}

In [ ]:
# 3 biến tạm này dở ẹt
day_of_month = df["date"].dt.day
day_to_month_end = df["date"].dt.days_in_month - day_of_month
dist_to_payday = np.minimum(day_of_month, day_to_month_end)  

# mae giảm 0.2%, tăng các cái còn lại, cần xem lại
# salary_heat = np.where(dist_to_payday <= 3, 1 / (dist_to_payday + 1), 0.0)
# is_salary_period = (dist_to_payday <= 3).astype(int)
# df["payday_weekend"] = (is_salary_period & (df["day_of_week"] >= 4)).astype(int)

model_config = SklearnRegressorConfig(
    model_type="lightgbm"
)
results = train_validate_models(
    df,
    train_range=TRAIN_RANGE,
    predict_range=VALIDATION_RANGE,
    model_config=model_config
)
results["metrics"]

{'Revenue': {'mae': 637068.0480624258,
  'rmse': 881026.1576368026,
  'mape': 20.789695537096925,
  'smape': 22.320541083523167,
  'r2': 0.7229454188865048},
 'COGS': {'mae': 535794.5113709207,
  'rmse': 726216.383398468,
  'mape': 20.82086101921165,
  'smape': 21.41569957930311,
  'r2': 0.752098537235178}}